# 14 — Three-seed fair method comparison

This is the paper-facing comparison workflow. It creates one immutable Stage-A checkpoint per seed, runs the full initialization/capacity/compute matrix, writes `Resource_Accounting.csv` for every condition, and aggregates paired seeds 0/1/2 into `Seed_Summary.csv`. Historical checkpoints without the Stage-A metadata contract are intentionally rejected.

In [ ]:
from pathlib import Path
import subprocess, sys

CONFIG = 'config/default.yaml'
PREFIX = 'fair_comparison_v1'
EXECUTION_MODE = 'out_of_core'  # out_of_core | in_memory
EXECUTE = False  # first inspect the printed 57-command matrix
SUMMARY_DIR = ''  # optional explicit output directory

command = [sys.executable, '-m', 'training.fairness_matrix', '--config', CONFIG,
           '--prefix', PREFIX, '--execution-mode', EXECUTION_MODE]
if EXECUTE:
    command.append('--execute')
if SUMMARY_DIR:
    command.extend(['--summary-dir', SUMMARY_DIR])
print(' '.join(command))
subprocess.run(command, check=True)


## Paper tables

The headline table uses dataset-blind methods only. `no_fusion` is retained as an explicitly labelled ground-truth-dataset oracle. Report mean ± sample SD and paired deltas from `Seed_Summary.csv`; the three-seed Student-t intervals are descriptive, not significance claims. An active-parameter control may be called iso-FLOPs only if its separately reported FLOP residual also meets tolerance.

In [ ]:
import glob, os
import pandas as pd

if SUMMARY_DIR and os.path.isfile(os.path.join(SUMMARY_DIR, 'Seed_Summary.csv')):
    summary = pd.read_csv(os.path.join(SUMMARY_DIR, 'Seed_Summary.csv'))
    display(summary[summary['dataset'] == 'ALL'].sort_values('macro_f1_mean', ascending=False))
else:
    print('Set SUMMARY_DIR after execution to display the aggregate table.')
